In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PersonalExpenseMonitoring") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [2]:
from google.colab import files

uploaded = files.upload()

Saving expenses.csv to expenses.csv


In [3]:
df = spark.read.csv(
    "expenses.csv",
    header=True,
    inferSchema=True
)

df.show()

+-------+----------+-------------+------+
|user_id|      date|     category|amount|
+-------+----------+-------------+------+
|      1|2026-06-01|         Food|  $250|
|      1|2026-06-02|    Transport|  $100|
|      1|2026-06-05|     Shopping| $1500|
|      2|2026-06-03|         Food|  $350|
|      2|2026-06-04|        Bills| $2500|
|      3|2026-06-06|    Transport|  $200|
|      3|2026-06-07|Entertainment|  $800|
|      4|2026-06-08|   Healthcare| $1200|
|      4|2026-06-09|         Food|  $400|
|      5|2026-06-10|     Shopping| $3000|
|      5|2026-06-11|        Bills| $1800|
+-------+----------+-------------+------+



In [5]:
from pyspark.sql.functions import regexp_replace, col

df = df.withColumn(
    "amount",
    regexp_replace(col("amount"), "\\$", "")
)

df = df.withColumn(
    "amount",
    col("amount").cast("double")
)

df.show()
df.printSchema()

+-------+----------+-------------+------+
|user_id|      date|     category|amount|
+-------+----------+-------------+------+
|      1|2026-06-01|         Food| 250.0|
|      1|2026-06-02|    Transport| 100.0|
|      1|2026-06-05|     Shopping|1500.0|
|      2|2026-06-03|         Food| 350.0|
|      2|2026-06-04|        Bills|2500.0|
|      3|2026-06-06|    Transport| 200.0|
|      3|2026-06-07|Entertainment| 800.0|
|      4|2026-06-08|   Healthcare|1200.0|
|      4|2026-06-09|         Food| 400.0|
|      5|2026-06-10|     Shopping|3000.0|
|      5|2026-06-11|        Bills|1800.0|
+-------+----------+-------------+------+

root
 |-- user_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)



In [6]:
from pyspark.sql.functions import col, sum, date_format

monthly_spend = (
    df.withColumn(
        "month",
        date_format(col("date"), "yyyy-MM")
    )
    .groupBy("user_id", "month")
    .agg(
        sum("amount").alias("total_monthly_spend")
    )
)

monthly_spend.show()

+-------+-------+-------------------+
|user_id|  month|total_monthly_spend|
+-------+-------+-------------------+
|      3|2026-06|             1000.0|
|      5|2026-06|             4800.0|
|      4|2026-06|             1600.0|
|      1|2026-06|             1850.0|
|      2|2026-06|             2850.0|
+-------+-------+-------------------+



In [7]:
from pyspark.sql.functions import avg

avg_expense = (
    df.agg(
        avg("amount").alias("avg_amount")
    )
)

avg_expense.show()

+----------+
|avg_amount|
+----------+
|    1100.0|
+----------+



In [8]:
avg_amount = (
    df.agg(avg("amount"))
    .collect()[0][0]
)

large_expenses = (
    df.filter(
        col("amount") > avg_amount * 3
    )
)

large_expenses.show()

+-------+----+--------+------+
|user_id|date|category|amount|
+-------+----+--------+------+
+-------+----+--------+------+



In [9]:
from pyspark.sql.functions import avg

user_avg = (
    df.groupBy("user_id")
    .agg(
        avg("amount").alias("user_avg")
    )
)

user_avg.show()

+-------+-----------------+
|user_id|         user_avg|
+-------+-----------------+
|      1|616.6666666666666|
|      3|            500.0|
|      5|           2400.0|
|      4|            800.0|
|      2|           1425.0|
+-------+-----------------+



In [10]:
spending_spikes = (
    df.join(user_avg, "user_id")
    .filter(
        col("amount") > col("user_avg") * 2
    )
)

spending_spikes.show()

+-------+----------+--------+------+-----------------+
|user_id|      date|category|amount|         user_avg|
+-------+----------+--------+------+-----------------+
|      1|2026-06-05|Shopping|1500.0|616.6666666666666|
+-------+----------+--------+------+-----------------+



In [11]:
top_users = (
    df.groupBy("user_id")
    .agg(
        sum("amount").alias("total_spend")
    )
    .orderBy(
        col("total_spend").desc()
    )
)

top_users.show()

+-------+-----------+
|user_id|total_spend|
+-------+-----------+
|      5|     4800.0|
|      2|     2850.0|
|      1|     1850.0|
|      4|     1600.0|
|      3|     1000.0|
+-------+-----------+



In [12]:
category_analysis = (
    df.groupBy("category")
    .agg(
        sum("amount").alias("total_amount")
    )
    .orderBy(
        col("total_amount").desc()
    )
)

category_analysis.show()

+-------------+------------+
|     category|total_amount|
+-------------+------------+
|     Shopping|      4500.0|
|        Bills|      4300.0|
|   Healthcare|      1200.0|
|         Food|      1000.0|
|Entertainment|       800.0|
|    Transport|       300.0|
+-------------+------------+

